---
## 순환 그래프 — AI끼리 대화 시켜보기

15.2의 **A ↔ B ↔ judge** 순환 구조에 LangChain LLM을 넣습니다.

| Node | 역할 |
|------|------|
| `optimist` | 낙관론자 — 주제에 찬성·긍정 |
| `skeptic` | 회의론자 — 반박·비판 |

**중단 조건** (`debate_route`):
* `turn_count >= max_turns` → `END`
* 마지막 메시지에 `'패배 인정'` 포함 → `END`
* 그 외 → `optimist`로 돌아가 순환

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)

In [ ]:
from typing import Literal, Annotated
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    max_turns: int = 3
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'


llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.8, api_key=OPENAI_API_KEY)

In [ ]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제는 {state.topic} 이제부터는 사람 없이 AI끼리 토론을 진행합니다.')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
             "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

### `route` 함수와 Conditional Edge 구현
* `debate_route`는 **다음에 갈 Node 이름** 또는 `END`를 반환합니다.
* `add_conditional_edges('skeptic', debate_route)`

In [ ]:
from langgraph.graph import StateGraph, START, END

def debate_route(state: DebateState):
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '패배 인정' in last_text:
        return END
    return 'optimist'


debate_workflow = StateGraph(DebateState)
debate_workflow.add_node('optimist', optimist_node)
debate_workflow.add_node('skeptic', skeptic_node)

debate_workflow.add_edge(START, 'optimist')
debate_workflow.add_edge('optimist', 'skeptic')
debate_workflow.add_conditional_edges('skeptic', debate_route)

debate_app = debate_workflow.compile()

In [ ]:
init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.')

for chunk in debate_app.stream(init_debate):
    print(chunk)

## 사회자 추가하기

토론 그래프에 **`moderator` Node**를 추가해 보세요.

* 매 라운드 끝에 양쪽 주장을 한 줄로 요약
* `debate_route`에서 `moderator`를 거친 뒤 `optimist`로 보내기
* 사회자가 '종료'를 언급해야 토론이 종료되도록 종료 조건 수정

In [ ]:
class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'

In [ ]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history
    ]
    

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

In [ ]:
def moderator_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 AI 토론 대회의 사회자입니다."
            "'찬성', '반대'측 AI 토론자 사이에서 토론을 진행하세요."
            "이전 발언자의 발언을 요약하세요"
            "토론을 종료하려면 '종료'를 언급하세요. 토론이 너무 길어지면 강제로 '종료'를 외쳐 토론을 마무리합니다."
        ))
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제는 {state.topic} 이제부터는 사람 없이 AI끼리 토론을 진행합니다.')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화
    
    ... #이어서 작성해보세요

In [ ]:
workflow = StateGraph(DebateState)

workflow.add_node("optimist", optimist_node)
workflow.add_node("skeptic", skeptic_node)
workflow.add_node("moderator", moderator_node)

...
# 이어서 edge 구현 후 실행해 보세요. 
# START에서 다음 노드가 어떤 노드여야할지 생각해 보세요.